# Week 10 — Scaling

Buying the search · SOC4180 Robot and AI

Hong Jeong

<figure>
<a
href="https://colab.research.google.com/github/gnoejh/soc4180/blob/main/weeks/w10-scaling/lab.ipynb"><img
src="https://colab.research.google.com/assets/colab-badge.svg" /></a>
<figcaption>Open In Colab</figcaption>
</figure>

**Before you start, two things:**

1.  **Runtime → Change runtime type → T4 GPU.** Required this week for
    more than rendering — the final section needs a GPU.
2.  **File → Save a copy in Drive.** This notebook is opened from GitHub
    and is *not* saved. Without a copy, your work disappears when you
    close the tab.

## The number we have been avoiding

Three weeks of training runs, and every one ended the same way: *not
enough steps*.

- Week 8: 30k steps. Learned not to fall.
- Week 9: 40k steps. Left one local optimum for a worse one.

A tuned locomotion run in the literature uses **150 million** steps.

We have been running **0.03%** of a real experiment and drawing
conclusions from it. Today we find out what that costs and how it is
bought.

------------------------------------------------------------------------

## Where the time goes

Not in the neural network. In the physics.

In [1]:
try:
    import soc4180
    import torch
except ImportError:
    %pip install -q "soc4180[rl] @ git+https://github.com/gnoejh/soc4180.git"
    import soc4180, torch

try:
    import mujoco_playground
except ImportError:
    # `playground`, NOT `playground[all]`. The [all] extra depends on
    # jax[cuda12] and would reinstall JAX over Colab's preinstalled GPU build,
    # silently leaving you on CPU. Plain `playground` leaves JAX alone.
    %pip install -q playground
    import mujoco_playground

import time, warnings
import numpy as np
warnings.filterwarnings("ignore")
from soc4180.envs import G1WalkEnv

env = G1WalkEnv(); env.reset(seed=0)
action = np.zeros(12, dtype=np.float32)

t0 = time.time(); n = 3000
for _ in range(n):
    _, _, terminated, truncated, _ = env.step(action)
    if terminated or truncated:
        env.reset()
single = n / (time.time() - t0)

print(f"one environment : {single:8.0f} control steps/s")
print(f"                  {single*env.decimation:10,.0f} physics steps/s "
      f"({env.decimation} per control step)")
print(f"150M steps at this rate: {150e6/single/3600:6.1f} hours "
      f"({150e6/single/86400:.1f} days)")

one environment :     1256 control steps/s
                      12,563 physics steps/s (10 per control step)
150M steps at this rate:   33.2 hours (1.4 days)

**More than a day of compute for one run.** And a real study needs
several seeds.

------------------------------------------------------------------------

## The obvious idea, and why it fails

Run many environments at once. Stable-Baselines3 offers `DummyVecEnv`:

In [2]:
from stable_baselines3.common.vec_env import DummyVecEnv

for k in (1, 4, 16):
    vec = DummyVecEnv([lambda: G1WalkEnv() for _ in range(k)])
    vec.reset(); acts = np.zeros((k, 12), dtype=np.float32)
    t0 = time.time(); m = 400
    for _ in range(m):
        vec.step(acts)
    print(f"  DummyVecEnv, {k:2d} envs : {m*k/(time.time()-t0):7.0f} steps/s")
    vec.close()

  DummyVecEnv,  1 envs :    1192 steps/s
  DummyVecEnv,  4 envs :    1250 steps/s
  DummyVecEnv, 16 envs :    1230 steps/s

Sixteen environments, **the same throughput as one**.

`DummyVecEnv` steps them *in a loop*. It batches the policy’s forward
pass, which was never the bottleneck. The physics still runs one
environment at a time on one core.

> “Vectorised” does not mean “parallel”.

------------------------------------------------------------------------

## Real parallelism, and its ceiling

`SubprocVecEnv` puts each environment in its own process, so the physics
really does run at the same time:

In [3]:
from stable_baselines3.common.vec_env import SubprocVecEnv
import os

print(f"CPU cores available here: {os.cpu_count()}\n")
for k in (2, 4, 8):
    vec = SubprocVecEnv([G1WalkEnv for _ in range(k)])
    vec.reset(); acts = np.zeros((k, 12), dtype=np.float32)
    for _ in range(10):
        vec.step(acts)                      # warm up the workers
    t0 = time.time(); m = 300
    for _ in range(m):
        vec.step(acts)
    print(f"  SubprocVecEnv, {k} envs : {m*k/(time.time()-t0):7.0f} steps/s")
    vec.close()

CPU cores available here: 36

  SubprocVecEnv, 2 envs :    1590 steps/s
  SubprocVecEnv, 4 envs :    3108 steps/s
  SubprocVecEnv, 8 envs :    5575 steps/s

This scales — until you run out of cores. **A Colab CPU runtime has
two.**

Even at a perfect 8× we would still need four hours per run, and the
tuned config asks for **8192** environments. There are not enough cores
in the building.

------------------------------------------------------------------------

## Physics on the GPU

The way out is not more cores. It is a different kind of processor.

**MJX** is MuJoCo rewritten in JAX. The same model, the same contact
solver — but expressed so that XLA can compile it and run **thousands of
copies at once** on a GPU.

Two ideas make it work:

- **`jit`** — compile the whole step function once, ahead of time,
  instead of interpreting Python 500 times a second
- **`vmap`** — take a function written for *one* robot and vectorise it
  over a batch dimension, so 8192 robots step in one GPU kernel launch

The price is real: everything must be expressible as fixed-shape array
operations. **No Python branching, no variable-length loops, no early
exit** — which is why an MJX environment reads so differently from the
one we wrote.

------------------------------------------------------------------------

## What the professionals actually run

`mujoco_playground` ships tuned configurations. These are read from the
library, not quoted from memory:

In [4]:
from mujoco_playground.config import locomotion_params

for name in ("BerkeleyHumanoidJoystickFlatTerrain", "Op3Joystick"):
    cfg = locomotion_params.brax_ppo_config(name)
    print(f"{name}")
    print(f"    timesteps   {cfg['num_timesteps']:>12,}")
    print(f"    parallel envs {cfg['num_envs']:>10,}")
    print(f"    batch size  {cfg['batch_size']:>12}")
    print(f"    discounting {cfg['discounting']:>12}")

BerkeleyHumanoidJoystickFlatTerrain
    timesteps    150,000,000
    parallel envs      8,192
    batch size           256
    discounting         0.97
Op3Joystick
    timesteps    100,000,000
    parallel envs      8,192
    batch size           256
    discounting         0.97

8192 environments in parallel, 150 million steps. Our entire Week 9
experiment was **0.027%** of one of these.

------------------------------------------------------------------------

## A detail worth noticing

In [5]:
cfg = locomotion_params.brax_ppo_config("BerkeleyHumanoidJoystickFlatTerrain")
net = dict(cfg["network_factory"])
for k, v in net.items():
    print(f"  {k:28s} {v}")

  policy_hidden_layer_sizes    (512, 256, 128)
  policy_obs_key               state
  value_hidden_layer_sizes     (512, 256, 128)
  value_obs_key                privileged_state

Look at the last two lines. The **policy** reads `state`; the **value
function** reads `privileged_state`.

That is the asymmetric actor–critic from Week 6, in production. The
critic is allowed to see things no real robot can measure — body height,
contact forces, terrain — because the critic is thrown away after
training. The policy sees only what an IMU and joint encoders provide,
so it remains deployable.

**The discipline we imposed in Week 6 is exactly what the state of the
art does.**

------------------------------------------------------------------------

## Running it

This needs a **GPU runtime**, and it will not run on a Windows machine
at all — JAX publishes CUDA wheels for Linux only.

In [6]:
import functools
from mujoco_playground import registry, wrapper
from mujoco_playground.config import locomotion_params
from brax.training.agents.ppo import train as ppo
from brax.training.agents.ppo import networks as ppo_networks

env_name = "BerkeleyHumanoidJoystickFlatTerrain"

# impl="jax" picks the MJX backend. The config default is "warp" (MJWarp), which
# needs the separate `mujoco-warp` package; without it registry.load dies with
#   AttributeError: type object 'int' has no attribute 'WARP'
env = registry.load(env_name, config_overrides={"impl": "jax"})

params = dict(locomotion_params.brax_ppo_config(env_name))
net = params.pop("network_factory")
params["network_factory"] = functools.partial(ppo_networks.make_ppo_networks, **net)
params.update(num_timesteps=15_000_000,   # 10% of the tuned budget
              num_envs=2048,              # a T4 has less memory than the target device
              num_evals=5)

make_inference, weights, metrics = ppo.train(
    environment=env,
    # a playground env is not a brax env; this adapts it
    wrap_env_fn=wrapper.wrap_for_brax_training,
    **params,
)

`registry.load` clones its **own** copy of Menagerie on first call —
about 40 seconds, separate from the copy `soc4180` uses.

------------------------------------------------------------------------

## Three things that will bite you

Every one of these was hit while preparing this lab.

**1. `ModuleNotFoundError: mujoco_playground`.** It is not in
`soc4180[rl]`. Install plain `playground` — **never `playground[all]`**,
whose `jax[cuda12]` dependency reinstalls JAX over Colab’s GPU build and
silently drops you to CPU.

**2. `AttributeError: type object 'int' has no attribute 'WARP'`.** The
env config defaults to `impl="warp"`, the newer MJWarp backend, which
needs the separate `mujoco-warp` package. Override to `impl="jax"`
(verified working) or install `mujoco-warp`.

**3. `AttributeError: jax.device_put_replicated is deprecated`.** brax
0.14.2 calls an API that JAX 0.11 removed, and brax requires only
`jax>=0.4.6` with no upper bound — so pip will happily install an
incompatible pair. Colab’s preinstalled JAX is older and works. **Do not
upgrade JAX on Colab.**

Two of these produce a traceback pointing deep inside someone else’s
library, and none of them is your code being wrong. **Version drift is
the normal state of this ecosystem**, and reading a stack trace back to
a version mismatch is a skill worth practising here rather than the week
before a deadline.

------------------------------------------------------------------------

## Track A and Track B

**Track A — required, ≤15 minutes.** Run the truncated configuration
above. Success is **not** a walking robot. Success is:

> episode reward and episode length both trending upward, and a rollout
> that shows a recognisable attempt at a gait.

**Track B — optional.** The full 150M-step budget, launched at the start
of the lab and collected at the end, or run overnight with checkpoints
saved to Drive.

Colab disconnects. **Checkpoint every evaluation**, or a run that dies
at minute 80 of 90 leaves you with nothing.

------------------------------------------------------------------------

## Reading a learning curve

You will be judged on the diagnosis, not the policy. Four shapes to
recognise:

| Curve | Likely cause |
|------------------------------------|------------------------------------|
| Flat from the start | broken observation, wrong action scale, or no gradient reaching the policy |
| Rises then collapses | learning rate too high, or the exploration bug from Week 8 |
| Rises then plateaus below target | local optimum — Week 9’s standing robot |
| Rises smoothly, slowly | it is working; you need more steps |

Always plot **episode length** alongside reward. A rising reward with
flat episode length usually means the policy found a way to score
without surviving — and that is a reward bug, not a training one.

------------------------------------------------------------------------

## What the budget actually buys

| Budget | What we saw |
|------------------------------------|------------------------------------|
| 30k steps (Week 8) | learns not to fall; scores the standing baseline |
| 40k steps (Week 9) | leaves the standing optimum, falls over backwards |
| 15M steps (today, Track A) | a recognisable gait attempt |
| 150M steps (tuned) | robust joystick-controlled walking |

Three orders of magnitude separate the top row from the bottom, and
nothing else about the algorithm changes. **PPO was never the
limitation.**

Our conclusions from Weeks 8 and 9 were not wrong — but they were
conclusions about *underfunded* runs, and it is worth being honest about
which findings survive a real budget.

------------------------------------------------------------------------

## Exercises

1.  **Measure your own ceiling.** Run the `SubprocVecEnv` sweep on
    Colab. How many cores do you get, and where does throughput stop
    improving?
2.  **Do the arithmetic.** At your measured CPU rate, how long is 150M
    steps? How many GPU-equivalents of CPU would you need?
3.  **Track A.** Run the truncated MJX configuration. Plot reward *and*
    episode length. Classify the curve using the table above.
4.  **Halve the environments.** Set `num_envs` to 1024 and then 4096 at
    a fixed step budget. Does wall-clock scale as you expect? What does
    that say about where the GPU time goes?
5.  **Break it deliberately.** Set the learning rate to 3e-2 and
    describe the curve’s shape before looking.
6.  **Checkpoint discipline.** Write a callback that saves to Drive
    every evaluation and reloads on restart. Test it by interrupting the
    run.

------------------------------------------------------------------------

## Next week

**Week 11 — domain randomization.** With a real budget available, the
question changes from *can it learn* to *will it survive anything other
than the exact simulator it learned in*.

We randomise mass, friction, and actuation during training, and measure
what robustness costs.